# exp_002 · 트랙 A — 변이 유형 집계 피처

- **전역 실험 ID**: `iljun-logreg-002`  ·  **폴더**: `exp_002_variant_type` (GIT_STRATEGY §9.1)
- **Owner**: member_d (iljun)
- **기준선**: `member-d-logreg-001` = Macro F1 **0.36305** (소급 변경 안 함 · §9.1)

## 이 폴더의 파일이 하는 일

| 파일 | 역할 | 실행 |
|---|---|---|
| `preprocessing/preprocess.py` | 팀 인터페이스 `fit/transform` (features_A 를 감쌈) | common 이 부름 |
| `training/model.py` `run.py` | 팀 방식 학습 (**holdout**) | `python3 -m ...training.run` |
| `pipeline.py` | 내 **CV·게이트·지문** 검증 | `python3 .../pipeline.py` |
| `experiment.ipynb` *(이 파일)* | 탐색·ablation | 매번 |

## 다중 seed — 이번 판의 핵심 변경

이전 판은 **seed 하나**로 돌려서, 블록 하나가 만든 `+0.002` 같은 차이가
**실제 효과인지 폴드 분할이 흔들린 것인지 구분할 수 없었다.**

팀 【전처리 베이스라인】에 3-seed 값이 올라왔다 — `Logistic 0.33738 ± 0.00625`.
같은 잣대로 재기 위해 팀 `common/preprocessing_benchmark.py` 와 **동일한 프로토콜**을 쓴다.

```
cv_seeds   = (42, 52, 62)   ← 폴드 분할만 바꾼다
model_seed = 42             ← 모델 random_state 는 고정
```

둘을 하나로 묶으면 '분할 변동'과 '모델 변동'이 섞여 σ 를 해석할 수 없다.

> ⚠️ **σ 는 seed 3개에서 나온 표본표준편차다.** n=3 의 표준편차는 그 자체로 크게
> 부정확하다. 정밀한 상수처럼 쓰지 말고 "이 정도 흔들린다"는 눈금으로만 쓴다.

> ⏱ Run All 기준 **12\~18분** (CV 실행 횟수가 3배로 늘었다).


## Section 1 · 설정

In [7]:
import sys, platform, time, importlib
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import f1_score


def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "configs" / "baseline.yaml").exists():
            return p
    raise FileNotFoundError("레포 루트를 못 찾음. 저장소 안에서 실행하세요.")


ROOT = find_root(Path.cwd())
EXP = ROOT / "experiments" / "member_d" / "exp_002_variant_type"
ART = EXP / "artifacts"; ART.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(EXP)); sys.path.insert(0, str(ROOT))

import features_A as fa
import pipeline as pa
importlib.reload(fa); importlib.reload(pa)
from features_A import KINDS, classify, parse_sample_counts, fit_spec, build_features

CFG = pa.load_cfg(); P = CFG["pipeline"]
REF = P["baseline"]; REF_F1, REF_ACC = REF["f1_macro"], REF["accuracy"]
DEC = P["decimals"]; N_SPLITS = P["cv"]["n_splits"]
CV_SEEDS = tuple(P["cv"]["seeds"])       # 폴드 분할 seed — 팀과 동일 (42, 52, 62)
SEED = P["cv"]["model_seed"]             # 모델 random_state — 고정
TARGET, ID = "SUBCLASS", "ID"

TEAM_BENCH = {"logistic_3seed": (0.33738, 0.00625),   # 팀 【전처리 베이스라인】
              "lightgbm_3seed": (0.28992, 0.00065)}


def verdict(f1): return pa.verdict(f1, REF_F1, DEC)
def diff(f1): return f"{f1 - REF_F1:+.5f}"


print(f"experiment {CFG['experiment']['id']} · python {platform.python_version()}")
print(f"features_A {fa.__version__}  sha {pa.sha256(fa.__file__)[:12]}")
print(f"pipeline   {pa.PIPELINE_VERSION}  sha {pa.sha256(pa.__file__)[:12]}")
print(f"기준선 {REF['experiment']} F1 {REF_F1:.5f} → 판정값 {round(REF_F1, DEC)}")
print(f"cv_seeds {list(CV_SEEDS)} · model_seed {SEED} · StratifiedKFold-{N_SPLITS}")
print(f"팀 3-seed Logistic 참고: {TEAM_BENCH['logistic_3seed'][0]:.5f} "
      f"± {TEAM_BENCH['logistic_3seed'][1]:.5f}")

experiment iljun-logreg-002 · python 3.12.13
features_A A_v1_variant_type  sha d164d6e17c32
pipeline   exp002_v2  sha 9856c8874893
기준선 member-d-logreg-001 F1 0.36305 → 판정값 0.363
cv_seeds [42, 52, 62] · model_seed 42 · StratifiedKFold-5
팀 3-seed Logistic 참고: 0.33738 ± 0.00625


### 데이터·파싱

In [8]:
train, test, submission, gene_cols = pa.load_data(ROOT)
y_all = train[TARGET].values
cnt_train, cnt_test = pa.parse_all(train, test, gene_cols)

[Step 1] train (6201, 4386) · test (2546, 4385) · 유전자 4384
[Step 2] 파싱 5s


## Section 2 · 파서 확인

<callout>

**① 칸 내부 중복 토큰 1개로.** `"R248Q R248Q"` → 1건 (총 6,100건).
**② 판정 순서 고정.** `>` → `fs` → `del`/`ins` → 나머지. `^([A-Z*]+)(\d+)(.*)$`.
`TP469fs` 같은 두 글자 접두가 빠지지 않게 앞 아미노산을 여러 글자로 받는다.

</callout>

In [9]:
CASES = [("R248Q","missense"),("R248R","silent"),("R248*","nonsense"),
         ("TP469fs","frameshift"),("K57del","indel"),("468_469LG>F*","other"),("*261*","other")]
for tok, want in CASES:
    assert classify(tok) == want, f"{tok}: {classify(tok)} != {want}"
print(f"파서 단위 테스트 {len(CASES)}건 PASS\n")

tot = cnt_train[KINDS].sum().astype("int64")
print(pd.DataFrame({"건수": tot, "비중 %": (tot/tot.sum()*100).round(2)}).to_string())
print(f"\n합계 {int(tot.sum()):,}  (홍주님 표준안: missense 161,051 · silent 64,844)")

파서 단위 테스트 7건 PASS

                건수   비중 %
missense    161051  64.66
silent       64844  26.04
nonsense     13080   5.25
frameshift    9764   3.92
indel            3   0.00
other          322   0.13

합계 249,064  (홍주님 표준안: missense 161,051 · silent 64,844)


## Section 3 · 피처 블록

| 블록 | 내용 | 차원 |
|---|---|---|
| **G** | 유전자 이진화 (fold 상수열 제거) | ~4,230 |
| **B** | 변이 부담 log1p (유전자·이벤트·다중) | 3 |
| **V** | 유형 카운트 log1p | 6 |
| **R** | 유형 비율 | 6 |

모두 행 내부 연산이라 Leakage 아님. 배제한 것은 `features_A.py` docstring 참고.

In [10]:
spec = fit_spec(train, gene_cols, seed=SEED)
X, names = build_features(train, cnt_train, spec)
print(f"전체 블록 {X.shape} 밀도 {X.nnz/np.prod(X.shape)*100:.2f}% · 상수열 {len(gene_cols)}→{len(spec['keep_idx'])}")
print("피처 예시:", names[:2], "...", names[-3:])

전체 블록 (6201, 4245) 밀도 1.02% · 상수열 4384→4230
피처 예시: ['A_gene__A2M', 'A_gene__AAAS'] ... ['A_vratio__frameshift', 'A_vratio__indel', 'A_vratio__other']


## Section 4 · Leakage 자가검증

In [11]:
ok, _ = pa.leakage_checks(train, test, cnt_test, gene_cols, tuple(P["blocks"]), SEED)
assert ok

         [PASS] 부분집합 불변성 (앞 100행)
         [PASS] 단일 행 독립성
         [PASS] spec 재현성
         [PASS] NaN·inf 없음
         [PASS] test 결측 fillna 처리


## Section 5 · Ablation (다중 seed)

각 조합을 `cv_seeds` 3개로 돌려 **평균 ± 표준편차**를 낸다.
CV 9회 × 5 fold 라 시간이 걸린다 (10분 안팎).

In [12]:
ABLATION = [("G",   "G        유전자만"),
            ("GB",  "G+B      + 변이 부담"),
            ("GBV", "G+B+V    + 유형 카운트"),
            ("GBVR","G+B+V+R  + 유형 비율"),
            ("BVR", "B+V+R    유전자 없이")]

results = []
for b, l in ABLATION:
    r = pa.cross_validate_multi(train, y_all, cnt_train, gene_cols, tuple(b),
                                model_key="logreg", cv_seeds=CV_SEEDS,
                                model_seed=SEED, n_splits=N_SPLITS, v=False)
    r["label"] = l; results.append(r)
    per = "  ".join(f"{d['f1_macro']:.5f}" for d in r["per_seed"])
    print(f"{l:28} dim {r['dim']:5d}  F1 {r['f1_macro']:.5f} ± {r['f1_macro_std']:.5f}  "
          f"({diff(r['f1_macro'])})  [seed별 {per}]")

G        유전자만                dim  4226  F1 0.33717 ± 0.00656  (-0.02588)  [seed별 0.34469  0.33260  0.33422]
G+B      + 변이 부담             dim  4229  F1 0.36325 ± 0.01036  (+0.00020)  [seed별 0.37429  0.35374  0.36173]
G+B+V    + 유형 카운트            dim  4235  F1 0.37713 ± 0.00512  (+0.01408)  [seed별 0.38218  0.37195  0.37727]
G+B+V+R  + 유형 비율             dim  4241  F1 0.37806 ± 0.00555  (+0.01501)  [seed별 0.38389  0.37283  0.37746]
B+V+R    유전자 없이              dim    15  F1 0.15690 ± 0.00096  (-0.20615)  [seed별 0.15581  0.15762  0.15727]


### 5-1 · 블록 하나를 더했을 때의 순증분 — **paired 비교**

**평균 증분 자체는 두 평균을 그냥 빼도 같은 값이 나온다.** paired 로 얻는 것은
그 증분의 **불확실성 추정**이다 — 같은 `cv_seed` 끼리 짝지으면 두 조합에 공통으로
실린 폴드 분할 변동이 상쇄되어, 증분의 σ 가 각 조합의 σ 보다 작아진다.
그래서 "이 증분이 흔들림 안에 있는가"를 훨씬 예민하게 볼 수 있다.

각 증분에 대해 아래 셋을 함께 본다.

- `평균 증분` — 효과의 크기
- `증분 σ` — 그 크기가 seed 에 따라 얼마나 흔들리는가
- `양수 seed` — 3개 중 몇 개에서 올랐는가 (**n=3 이라 3/3 이어도 통계적 유의는 아니다**)

In [13]:
def paired_delta(after, before):
    """같은 cv_seed 끼리 짝지어 뺀 증분. (평균, 표준편차, 양수 개수, 원본 배열)"""
    a = {d["cv_seed"]: d["f1_macro"] for d in after["per_seed"]}
    b = {d["cv_seed"]: d["f1_macro"] for d in before["per_seed"]}
    seeds = sorted(set(a) & set(b))
    d = np.array([a[s] - b[s] for s in seeds], dtype=float)
    sd = float(d.std(ddof=1)) if len(d) > 1 else float("nan")
    return float(d.mean()), sd, int((d > 0).sum()), d


by = {r["blocks"]: r for r in results}
STEPS = [("B  변이 부담",   "GB",   "G"),
         ("V  유형 카운트", "GBV",  "GB"),
         ("R  유형 비율",   "GBVR", "GBV")]

rows = []
for name, after, before in STEPS:
    m, sd, pos, d = paired_delta(by[after], by[before])
    rows.append({"추가한 블록": name, "평균 증분": round(m, 5),
                 "증분 σ": round(sd, 5), "양수 seed": f"{pos}/{len(d)}",
                 "|증분|/σ": round(abs(m) / sd, 1) if sd and sd == sd and sd > 0 else None})
steps_tab = pd.DataFrame(rows)
print(steps_tab.to_string(index=False))
print()
print("※ |증분|/σ 는 참고용 눈금이다. n=3 의 σ 는 그 자체로 부정확하므로")
print("  이 값으로 유의성을 주장하지 않는다. 방향과 대략적 크기만 읽는다.")
steps_tab.to_csv(ART/"ablation_steps.csv", index=False, encoding="utf-8-sig")

   추가한 블록   평균 증분    증분 σ 양수 seed  |증분|/σ
 B  변이 부담 0.02608 0.00441     3/3     5.9
V  유형 카운트 0.01388 0.00536     3/3     2.6
 R  유형 비율 0.00093 0.00076     3/3     1.2

※ |증분|/σ 는 참고용 눈금이다. n=3 의 σ 는 그 자체로 부정확하므로
  이 값으로 유의성을 주장하지 않는다. 방향과 대략적 크기만 읽는다.


In [14]:
tab = pd.DataFrame([{"label": r["label"], "dim": r["dim"],
                     "f1_macro": r["f1_macro"], "f1_std": r["f1_macro_std"],
                     "accuracy": r["accuracy"]} for r in results])
tab["기준선 대비"] = (tab["f1_macro"] - REF_F1).round(5)
tab["판정"] = [verdict(v) for v in tab["f1_macro"]]
print(tab.to_string(index=False))
tab.to_csv(ART/"ablation.csv", index=False, encoding="utf-8-sig")

lo, ls = TEAM_BENCH["logistic_3seed"]
print(f"\n팀 3-seed Logistic(이진화 전처리) {lo:.5f} ± {ls:.5f} 와 비교해 읽을 것.")

            label  dim  f1_macro  f1_std  accuracy   기준선 대비     판정
    G        유전자만 4226   0.33717 0.00656   0.33618 -0.02588 [-] 하락
 G+B      + 변이 부담 4229   0.36325 0.01036   0.35806  0.00020 [=] 동일
G+B+V    + 유형 카운트 4235   0.37713 0.00512   0.37134  0.01408 [+] 향상
 G+B+V+R  + 유형 비율 4241   0.37806 0.00555   0.37252  0.01501 [+] 향상
  B+V+R    유전자 없이   15   0.15690 0.00096   0.17395 -0.20615 [-] 하락

팀 3-seed Logistic(이진화 전처리) 0.33738 ± 0.00625 와 비교해 읽을 것.


## Section 6 · 어느 클래스가 좋아졌나

클래스별 F1 은 `cv_seeds[0]` (=42) 의 OOF 하나만 쓴다.
**따라서 이 표의 개별 클래스 변화에는 오차 막대가 없다.** 방향을 보는 용도로만 읽고,
결론을 내릴 때는 Section 5-1 의 증분과 함께 판단한다.

In [15]:
classes = sorted(pd.unique(y_all))
best = max(results, key=lambda r: r["f1_macro"])
refr = by.get("GB") if by.get("GB") is not best else by.get("G", best)
if refr is best:
    print("※ 최고 조합이 비교 기준과 같습니다.\n")

d = pd.DataFrame({refr["blocks"]: f1_score(y_all, refr["oof"], average=None, labels=classes),
                  best["blocks"]+" (최고)": f1_score(y_all, best["oof"], average=None, labels=classes)},
                 index=classes)
d["변화"] = (d.iloc[:,1]-d.iloc[:,0]).round(4); d = d.round(4).sort_values("변화", ascending=False)
print(f"최고 {best['label'].strip()} ({best['f1_macro']:.5f}) · 비교 {refr['blocks']} "
      f"· cv_seed {CV_SEEDS[0]} OOF 기준\n")
print("오른 8개"); print(d.head(8).to_string())
print("\n내린 5개"); print(d.tail(5).to_string())
W=["LAML","DLBC","ACC","SKCM","THYM"]; w=d.loc[[c for c in W if c in d.index]]
print("\n가설 대상 (유형 구성이 극단적인 클래스)"); print(w.to_string())
print(f"\n{len(w)}개 중 {int((w['변화']>0).sum())}개 상승 — seed 1개 기준이므로 참고값")
d.to_csv(ART/"class_f1_delta.csv", encoding="utf-8-sig")

최고 G+B+V+R  + 유형 비율 (0.37806) · 비교 GB · cv_seed 42 OOF 기준

오른 8개
          GB  GBVR (최고)      변화
LUSC  0.2884     0.4037  0.1153
CESC  0.1221     0.1805  0.0584
PAAD  0.1942     0.2466  0.0524
STES  0.3476     0.3848  0.0372
SARC  0.1721     0.2057  0.0336
LUAD  0.2353     0.2655  0.0302
BRCA  0.4842     0.5104  0.0262
OV    0.3477     0.3633  0.0157

내린 5개
          GB  GBVR (최고)      변화
THYM  0.2881     0.2763 -0.0118
KIRC  0.1326     0.1152 -0.0174
ACC   0.8421     0.8182 -0.0239
DLBC  0.4643     0.4231 -0.0412
BLCA  0.3399     0.2857 -0.0542

가설 대상 (유형 구성이 극단적인 클래스)
          GB  GBVR (최고)      변화
LAML  0.5423     0.5382 -0.0040
DLBC  0.4643     0.4231 -0.0412
ACC   0.8421     0.8182 -0.0239
SKCM  0.7181     0.7302  0.0122
THYM  0.2881     0.2763 -0.0118

5개 중 1개 상승 — seed 1개 기준이므로 참고값


## Section 7 · 모델 비교 (다중 seed)

`LinearSVC` 는 `predict_proba` 가 없어 팀 프레임워크(`validate_model_interface`)에서 쓸 수 없다.
여기서는 비교용으로만 돌린다.

In [16]:
bb = tuple(best["blocks"])
print(f"피처 고정 {best['blocks']}\n")
model_results = []
for k in ["logreg","svm","sgd"]:
    r = pa.cross_validate_multi(train, y_all, cnt_train, gene_cols, bb,
                                model_key=k, cv_seeds=CV_SEEDS,
                                model_seed=SEED, n_splits=N_SPLITS, v=False)
    model_results.append(r)
    print(f"{r['model_name']:32} F1 {r['f1_macro']:.5f} ± {r['f1_macro_std']:.5f}  "
          f"({diff(r['f1_macro'])})  Acc {r['accuracy']:.5f}")

피처 고정 GBVR

LogisticRegression(balanced)     F1 0.37806 ± 0.00555  (+0.01501)  Acc 0.37252
LinearSVC(balanced)              F1 0.30117 ± 0.00781  (-0.06188)  Acc 0.30619
SGD(modified_huber, balanced)    F1 0.31878 ± 0.00385  (-0.04427)  Acc 0.32333


## Section 8 · 파이프라인 교차검증 (검증 계약)

`pipeline.py` 를 CSV 부터 처음부터 돌려 **노트북과 같은 평균이 나오는지** assert 한다.
`--repeat` 로 같은 `cv_seed` 재실행 결정성도 확인한다.

In [17]:
champ = max(results, key=lambda r: r["f1_macro"])
res = pa.run_pipeline(root=ROOT, blocks=champ["blocks"], repeat=2)

same_f1  = res["f1_macro"] == champ["f1_macro"]
same_std = res["f1_macro_std"] == champ["f1_macro_std"]
print("\n노트북 vs 파이프라인")
print(f"  평균 Macro F1  {champ['f1_macro']:.5f} vs {res['f1_macro']:.5f}   "
      f"[{'PASS' if same_f1 else 'FAIL'}]")
print(f"  표준편차       {champ['f1_macro_std']:.5f} vs {res['f1_macro_std']:.5f}   "
      f"[{'PASS' if same_std else 'FAIL'}]")
print(f"  결정성 (같은 cv_seed 재실행)                [{'PASS' if res['deterministic'] else 'FAIL'}]")
assert same_f1 and same_std and res["deterministic"], "노트북↔파이프라인 불일치"

print()
print(f"{res['verdict']}  {res['experiment']}  "
      f"F1 {res['f1_macro']:.5f} ± {res['f1_macro_std']:.5f} ({res['delta_vs_baseline']:+.5f})")
print(f"지문 pipeline {res['fingerprint']['pipeline_sha256'][:12]} · "
      f"features {res['fingerprint']['features_sha256'][:12]}")

  iljun-logreg-002 · pipeline exp002_v2 · features_A A_v1_variant_type
  피처 GBVR · LogisticRegression(balanced) · model_seed 42 · StratifiedKFold-5 · cv_seeds [42, 52, 62]
  기준선 member-d-logreg-001 = 0.36305 (판정 0.363)
[Step 1] train (6201, 4386) · test (2546, 4385) · 유전자 4384
[Step 2] 파싱 2s
         [PASS] 부분집합 불변성 (앞 100행)
         [PASS] 단일 행 독립성
         [PASS] spec 재현성
         [PASS] NaN·inf 없음
         [PASS] test 결측 fillna 처리
[Step 4] 교차검증 · cv_seeds [42, 52, 62]
         GBVR                           cv_seed 42  dim  4241  F1 0.38389  Acc 0.37671  (29s)
         GBVR                           cv_seed 52  dim  4244  F1 0.37283  Acc 0.36849  (29s)
         GBVR                           cv_seed 62  dim  4244  F1 0.37746  Acc 0.37236  (33s)
         └ 평균                           F1 0.37806 ± 0.00555  Acc 0.37252
         결정성 재실행                        cv_seed 42  dim  4241  F1 0.38389  Acc 0.37671  (34s)
         결정성 PASS
  Macro F1 0.37806 ± 0.00555  (+0.01501)  Acc 0.37252  [

### ✅ 실행 후 읽는 법

**Section 5-1 이 이 노트북의 결론입니다.** 다음 순서로 읽습니다.

1. `B`(변이 부담) 의 평균 증분과 σ — 이전 단일 seed 판에서 가장 컸던 항목
2. `V`(유형 카운트) — **이 노트북의 원래 가설.** 단일 seed 에서 `+0.00789` 였으나
   그때는 σ 를 몰라 판단할 수 없었다. 이번에 처음으로 판단 근거가 생긴다
3. `R`(유형 비율) — 단일 seed 에서 `+0.00171`. σ 안에 묻히는지 확인
4. `양수 seed` 열 — 3/3 이면 방향은 일관되나, **n=3 이라 그것만으로 유의하지 않다**

**결론을 쓸 때 지킬 것**

- 증분이 σ 보다 작으면 "효과 없음"이 아니라 **"이 설계로는 검출하지 못함"** 으로 적는다
- σ 는 seed 3개짜리 추정이다. 배수를 정밀한 유의수준처럼 쓰지 않는다
- CV 에서 관찰된 것을 LB 로 확대하지 않는다 (제출 2건에서 CV > LB 가 관찰됐을 뿐,
  그 관계의 함수 형태는 아직 모른다)

**다음 단계** — Section 5-1 결과를 보고 정합니다.

```bash
# 노트북 없이 재현
python3 experiments/member_d/exp_002_variant_type/pipeline.py
python3 experiments/member_d/exp_002_variant_type/pipeline.py --smoke   # 30초 배선 점검
```
